In [23]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from typing import TypedDict
from dotenv import load_dotenv

In [24]:
load_dotenv()

True

In [25]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="conversational"
)

model = ChatHuggingFace(llm = llm)

In [26]:
# create a state
class LLMState(TypedDict) :
    question: str
    answer: str


In [27]:
def llm_qa(state: LLMState) -> LLMState:
    # extrace the question
    question = state['question']

    # form a prompt
    prompt = f'answer the following question {question}'

    # ask that question
    answer = model.invoke(prompt).content

    # update the answer
    state['answer'] = answer

    return state

In [28]:
# create graph
graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile the graph
workflow = graph.compile()

In [29]:
# execute
initial_state = {'question': 'How far is the moon from earth?'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'question': 'How far is the moon from earth?', 'answer': "On average, the moon is about 384,400 kilometers (238,900 miles) away from the Earth. However, this distance can vary slightly due to the elliptical shape of the moon's orbit around our planet. At its closest point (called perigee), the moon is about 363,300 kilometers (225,000 miles) away, and at its farthest point (apogee), it's about 405,500 kilometers (252,000 miles) away."}
